In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import yaml

import logging
import wandb

# Custom Utilities Module
from utils.paths import get_paths

from utils.file_io import (
    load_data, 
    save_data, 
    save_json, 
    load_json,
)

from utils.logging_setup import (
    configure_logging, 
    log_layer_paths,
)

from utils.wandb_utils import finalize_wandb_stage


from utils.pipeline_config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash,
    load_truth_record_by_hash,
    load_parent_truth_record_from_dataframe,
    get_truth_value,
    get_dataset_name_from_truth,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.core.cascade_row_tracking import (
    ensure_stable_row_id,
    get_detected_rows_dataframe,
    get_stage_detected_rows_dataframe,
)

from utils.postgres_util import get_engine_from_env

from utils.layer_postgres_writer import (
    write_layer_dataframe, 
    prepare_layer_dataframe,
)

# Ledger 
from utils.ledger import Ledger


# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)






In [ ]:
PLOT_CONTEXT = "cascade"   # "baseline" or "cascade"
ROW_ID_COLUMN = "meta__row_id"

# Preferred time axis resolution order
TIME_AXIS_CANDIDATES = [
    "event_time",
    "time_index",
    "event_step",
    ROW_ID_COLUMN,
]

# Sensor selection
DEFAULT_SENSOR_PREFIXES = ("sensor_",)
MAX_SENSORS_PER_BATCH = 6

# Marker styling
ANOMALY_MARKER_SIZE = 35
STAGE1_MARKER_SIZE = 20
STAGE2_MARKER_SIZE = 30
FINAL_MARKER_SIZE = 40

# Optional export
SAVE_PLOTS = False
PLOT_OUTPUT_DIR = Path("artifacts/gold/plots/time_series")
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_ROOT = paths.configs
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"

CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage="gold_anomaly_detection",
    dataset="pump",
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data

COMPARISON_CFG = CONFIG["gold_anomaly_detection"]
PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]
PIPELINE = CONFIG.get(
    "pipeline",
    {
        "execution_mode": "batch",
        "orchestration_mode": "notebook",
    },
)
RUN_MODE = CONFIG["runtime"]["mode"]

TRUTH_CONFIG = build_truth_config_block(CONFIG)
TRUTH_CONFIG["pipeline"] = PIPELINE
TRUTH_CONFIG["stage_params"] = COMPARISON_CFG

# Stage Details
STAGE = "gold"
LAYER_NAME = COMPARISON_CFG["layer_name"]
GOLD_VERSION = CONFIG["versions"]["gold"]
RECIPE_ID = COMPARISON_CFG["recipe_id"]
TRUTH_VERSION = CONFIG["versions"]["truth"]
PIPELINE_MODE = PIPELINE["execution_mode"]

DATASET_NAME_CONFIG = CONFIG["dataset"]["name"]
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#GOLD_PROCESS_RUN_ID = make_process_run_id(COMPARISON_CFG["process_run_id_prefix"])

# Weights and Biases
WANDB_PROJECT = CONFIG["wandb"]["project"]
WANDB_ENTITY = CONFIG["wandb"]["entity"]
WANDB_RUN_NAME = f"{GOLD_VERSION}"

# File names
# File names
BASELINE_RESULTS_FILE_NAME_CSV = FILENAMES["baseline_results_file_name_csv"]
BASELINE_RESULTS_FILE_NAME_PICKLE = FILENAMES["baseline_results_file_name_pickle"]
BASELINE_SUMMARY_FILE_NAME = FILENAMES["baseline_summary_file_name"]
BASELINE_THRESHOLDS_FILE_NAME = FILENAMES["baseline_thresholds_file_name"]
BASELINE_METADATA_FILE_NAME = FILENAMES["baseline_metadata_file_name"]

CASCADE_DEFAULTS_RESULTS_FILE_NAME_CSV = FILENAMES["cascade_defaults_results_file_name_csv"]
CASCADE_DEFAULTS_FILE_NAME_PICKLE = FILENAMES["cascade_defaults_results_file_name_pickle"]
CASCADE_DEFAULTS_SUMMARY_FILE_NAME = FILENAMES["cascade_defaults_summary_file_name"]
CASCADE_DEFAULTS_THRESHOLDS_FILE_NAME = FILENAMES["cascade_defaults_thresholds_file_name"]
CASCADE_DEFAULTS_METADATA_FILE_NAME = FILENAMES["cascade_defaults_metadata_file_name"]

CASCADE_TUNED_RESULTS_FILE_NAME_CSV = FILENAMES["cascade_tuned_results_file_name_csv"]
CASCADE_TUNED_RESULTS_FILE_NAME_PICKLE = FILENAMES["cascade_tuned_results_file_name_pickle"]
CASCADE_TUNED_SUMMARY_FILE_NAME = FILENAMES["cascade_tuned_summary_file_name"]
CASCADE_TUNED_THRESHOLDS_FILE_NAME = FILENAMES["cascade_tuned_thresholds_file_name"]
CASCADE_TUNED_METADATA_FILE_NAME = FILENAMES["cascade_tuned_metadata_file_name"]

CASCADE_STAGE3_IMPROVED_RESULTS_FILE_NAME_CSV = FILENAMES["cascade_stage3_improved_results_file_name_csv"]
CASCADE_STAGE3_IMPROVED_RESULTS_FILE_NAME_PICKLE = FILENAMES["cascade_stage3_improved_results_file_name_pickle"]
CASCADE_STAGE3_IMPROVED_SUMMARY_FILE_NAME = FILENAMES["cascade_stage3_improved_summary_file_name"]
CASCADE_STAGE3_IMPROVED_THRESHOLDS_FILE_NAME = FILENAMES["cascade_stage3_improved_thresholds_file_name"]
CASCADE_STAGE3_IMPROVED_METADATA_FILE_NAME = FILENAMES["cascade_stage3_improved_metadata_file_name"]

MODEL_COMPARISON_FILE_NAME = FILENAMES["model_comparison_file_name"]
MODEL_COMPARISON_SUMMARY_FILE_NAME = FILENAMES["model_comparison_summary_file_name"]

# File names
GOLD_PREPROCESSED_FILE_NAME = FILENAMES["gold_preprocessed_file_name"]
GOLD_PREPROCESSED_SCALED_FILE_NAME = FILENAMES["gold_preprocessed_scaled_file_name"]

GOLD_FIT_FILE_NAME = FILENAMES["gold_fit_file_name"]
GOLD_TRAIN_FILE_NAME = FILENAMES["gold_train_file_name"]
GOLD_TEST_FILE_NAME = FILENAMES["gold_test_file_name"]

GOLD_COMPARISON_LEDGER_FILE_NAME = FILENAMES["gold_comparison_ledger_file_name"]
COMPARISON_PLOT_WITH_TEST_ALERTS_FILE_NAME = FILENAMES["comparison_plot_with_test_alerts_file_name"]

set_wandb_dir_from_config(CONFIG)

GOLD_ARTIFACTS_PATH = Path(PATHS["gold_artifacts_dir"])

BASELINE_RESULTS_PATH_CSV = Path(PATHS["baseline_results_path_csv"])
BASELINE_RESULTS_PATH_PICKLE = Path(PATHS["baseline_results_path_pickle"])
BASELINE_THRESHOLDS_PATH = Path(PATHS["baseline_thresholds_path"])
BASELINE_SUMMARY_PATH = Path(PATHS["baseline_summary_path"])
BASELINE_METADATA_PATH = Path(PATHS["baseline_metadata_path"])

CASCADE_DEFAULTS_RESULTS_PATH_CSV = Path(PATHS["cascade_defaults_results_path_csv"])
CASCADE_DEFAULTS_RESULTS_PATH_PICKLE = Path(PATHS["cascade_defaults_results_path_pickle"])
CASCADE_DEFAULTS_THRESHOLDS_PATH = Path(PATHS["cascade_defaults_thresholds_path"])
CASCADE_DEFAULTS_SUMMARY_PATH = Path(PATHS["cascade_defaults_summary_path"])
CASCADE_DEFAULTS_METADATA_PATH = Path(PATHS["cascade_defaults_metadata_path"])

CASCADE_TUNED_RESULTS_PATH_CSV = Path(PATHS["cascade_tuned_results_path_csv"])
CASCADE_TUNED_RESULTS_PATH_PICKLE = Path(PATHS["cascade_tuned_results_path_pickle"])
CASCADE_TUNED_THRESHOLDS_PATH = Path(PATHS["cascade_tuned_thresholds_path"])
CASCADE_TUNED_SUMMARY_PATH = Path(PATHS["cascade_tuned_summary_path"])
CASCADE_TUNED_METADATA_PATH = Path(PATHS["cascade_tuned_metadata_path"])

CASCADE_STAGE3_IMPROVED_RESULTS_PATH_CSV = Path(PATHS["cascade_stage3_improved_results_path_csv"])
CASCADE_STAGE3_IMPROVED_RESULTS_PATH_PICKLE = Path(PATHS["cascade_stage3_improved_results_path_pickle"])
CASCADE_STAGE3_IMPROVED_THRESHOLDS_PATH = Path(PATHS["cascade_stage3_improved_thresholds_path"])
CASCADE_STAGE3_IMPROVED_SUMMARY_PATH = Path(PATHS["cascade_stage3_improved_summary_path"])
CASCADE_STAGE3_IMPROVED_METADATA_PATH = Path(PATHS["cascade_stage3_improved_metadata_path"])

CASCADE_STAGE3_RESULTS_PATH_CSV = PATHS["cascade_tuned_results_path_csv"]
CASCADE_STAGE3_RESULTS_PATH_PICKLE = PATHS["cascade_tuned_results_path_pickle"]
CASCADE_STAGE3_SUMMARY_PATH = PATHS["cascade_tuned_summary_path"]
CASCADE_STAGE3_THRESHOLDS_PATH = PATHS["cascade_tuned_thresholds_path"]
CASCADE_STAGE3_METADATA_PATH = PATHS["cascade_tuned_metadata_path"]

#MODEL_COMPARISON_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__model_comparison.csv"
#MODEL_COMPARISON_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / f"{DATASET_NAME}__gold__model_comparison_summary.json"

MODEL_COMPARISON_PATH = Path(PATHS["model_comparison_path"])
MODEL_COMPARISON_SUMMARY_PATH = Path(PATHS["model_comparison_summary_path"])

GOLD_PREPROCESSED_DATA_PATH = Path(PATHS["gold_preprocessed_data_path"])
GOLD_PREPROCESSED_SCALED_DATA_PATH = Path(PATHS["gold_preprocessed_scaled_data_path"])

GOLD_TRAIN_DATA_PATH = Path(PATHS["gold_train_data_path"])
GOLD_TEST_DATA_PATH = Path(PATHS["gold_test_data_path"])
GOLD_FIT_DATA_PATH = Path(PATHS["gold_fit_data_path"])

COMPARISON_PLOT_WITH_TEST_ALERTS_PATH = Path(PATHS["comparison_plot_with_test_alerts_path"]) 

# Logs
LOGS_PATH = Path(PATHS["logs_root"])

# Truths
TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])

# Path Failsafes


GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)




In [ ]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_model_anomaly_detection.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold Modeling stage starting")

# Log paths loads
log_layer_paths(paths, current_layer="gold", logger=logger)


In [ ]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_model_anomaly_detection",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
        "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),

        "cascade_defaults_results_path_csv": str(CASCADE_DEFAULTS_RESULTS_PATH_CSV),
        "cascade_defaults_results_path_pickle": str(CASCADE_DEFAULTS_RESULTS_PATH_PICKLE),
        "cascade_defaults_thresholds_path": str(CASCADE_DEFAULTS_THRESHOLDS_PATH),
        "cascade_defaults_metadata_path": str(CASCADE_DEFAULTS_METADATA_PATH),
        "cascade_defaults_summary_path": str(CASCADE_DEFAULTS_SUMMARY_PATH),

        "cascade_tuned_results_path_csv": str(CASCADE_TUNED_RESULTS_PATH_CSV),
        "cascade_tuned_results_path_pickle": str(CASCADE_TUNED_RESULTS_PATH_PICKLE),
        "cascade_tuned_thresholds_path": str(CASCADE_TUNED_THRESHOLDS_PATH),
        "cascade_tuned_metadata_path": str(CASCADE_TUNED_METADATA_PATH),
        "cascade_tuned_summary_path": str(CASCADE_TUNED_SUMMARY_PATH),

        "cascade_stage3_improved_results_path_csv": str(CASCADE_STAGE3_IMPROVED_RESULTS_PATH_CSV),
        "cascade_stage3_improved_results_path_pickle": str(CASCADE_STAGE3_IMPROVED_RESULTS_PATH_PICKLE),
        "cascade_stage3_improved_thresholds_path": str(CASCADE_STAGE3_IMPROVED_THRESHOLDS_PATH),
        "cascade_stage3_improved_metadata_path": str(CASCADE_STAGE3_IMPROVED_METADATA_PATH),
        "cascade_stage3_improved_summary_path": str(CASCADE_STAGE3_IMPROVED_SUMMARY_PATH),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)

In [ ]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


In [ ]:
baseline_results_csv = pd.read_csv(BASELINE_RESULTS_PATH_CSV)
baseline_results_pickle = pd.read_pickle(BASELINE_RESULTS_PATH_PICKLE)
baseline_results = baseline_results_csv

baseline_summary = load_json(BASELINE_SUMMARY_PATH)
baseline_thresholds = load_json(BASELINE_THRESHOLDS_PATH)
baseline_metadata = load_json(BASELINE_METADATA_PATH)

cascade_defaults_summary = load_json(CASCADE_DEFAULTS_SUMMARY_PATH)
cascade_defaults_thresholds = load_json(CASCADE_DEFAULTS_THRESHOLDS_PATH)
cascade_defaults_metadata = load_json(CASCADE_DEFAULTS_METADATA_PATH)
cascade_defaults_results_csv = pd.read_csv(CASCADE_DEFAULTS_RESULTS_PATH_CSV)
cascade_defaults_results_pickle = pd.read_pickle(CASCADE_DEFAULTS_RESULTS_PATH_PICKLE)
cascade_defaults_results = cascade_defaults_results_csv

cascade_tuned_summary = load_json(CASCADE_TUNED_SUMMARY_PATH)
cascade_tuned_thresholds = load_json(CASCADE_TUNED_THRESHOLDS_PATH)
cascade_tuned_metadata = load_json(CASCADE_TUNED_METADATA_PATH)
cascade_tuned_results_csv = pd.read_csv(CASCADE_TUNED_RESULTS_PATH_CSV)
cascade_tuned_results_pickle = pd.read_pickle(CASCADE_TUNED_RESULTS_PATH_PICKLE)
cascade_tuned_results = cascade_tuned_results_csv

cascade_stage3_improved_summary = load_json(CASCADE_STAGE3_IMPROVED_SUMMARY_PATH)
cascade_stage3_improved_thresholds = load_json(CASCADE_STAGE3_IMPROVED_THRESHOLDS_PATH)
cascade_stage3_improved_metadata = load_json(CASCADE_STAGE3_IMPROVED_METADATA_PATH)
cascade_stage3_improved_results_csv = pd.read_csv(CASCADE_STAGE3_IMPROVED_RESULTS_PATH_CSV)
cascade_stage3_improved_results_pickle = pd.read_pickle(CASCADE_STAGE3_IMPROVED_RESULTS_PATH_PICKLE)
cascade_stage3_improved_results = cascade_stage3_improved_results_csv
# cascade_stage3_improved_results = cascade_stage3_improved_results_pickle


gold_preprocessed_scaled_dataframe = load_data(GOLD_PREPROCESSED_SCALED_DATA_PATH)

# -------------------------------------------------------------------
# Resolve and validate Baseline stage truth
# -------------------------------------------------------------------
BASELINE_TRUTH_HASH = baseline_metadata.get("baseline_truth_hash")
BASELINE_TRUTH_PATH_VALUE = baseline_metadata.get("baseline_truth_path")

if BASELINE_TRUTH_HASH is None:
    raise ValueError("baseline_metadata is missing 'baseline_truth_hash'.")
if BASELINE_TRUTH_PATH_VALUE is None:
    raise ValueError("baseline_metadata is missing 'baseline_truth_path'.")

BASELINE_TRUTH_PATH = Path(BASELINE_TRUTH_PATH_VALUE)
if not BASELINE_TRUTH_PATH.exists():
    raise FileNotFoundError(f"Baseline truth file not found: {BASELINE_TRUTH_PATH}")

baseline_truth = load_json(BASELINE_TRUTH_PATH)

if baseline_truth.get("truth_hash") != BASELINE_TRUTH_HASH:
    raise ValueError(
        "Baseline metadata truth hash does not match the loaded baseline truth file:\n"
        f"metadata={BASELINE_TRUTH_HASH}\n"
        f"truth_file={baseline_truth.get('truth_hash')}"
    )

baseline_results_truth_hash = extract_truth_hash(baseline_results)
if baseline_results_truth_hash is None:
    raise ValueError("Could not resolve meta__truth_hash from baseline_results CSV.")

if baseline_results_truth_hash != BASELINE_TRUTH_HASH:
    raise ValueError(
        "Baseline results CSV truth hash does not match baseline_metadata['baseline_truth_hash']:\n"
        f"csv={baseline_results_truth_hash}\n"
        f"metadata={BASELINE_TRUTH_HASH}"
    )

BASELINE_PARENT_GOLD_TRUTH_HASH = get_parent_truth_hash(baseline_truth)
if BASELINE_PARENT_GOLD_TRUTH_HASH is None:
    raise ValueError("baseline_truth is missing a usable parent_truth_hash.")

baseline_truth_runtime_facts = baseline_truth.get("runtime_facts", {})
baseline_truth_artifact_paths = baseline_truth.get("artifact_paths", {})

# -------------------------------------------------------------------
# Resolve and validate Cascade Defaults stage truth
# -------------------------------------------------------------------
CASCADE_DEFAULTS_TRUTH_HASH = cascade_defaults_metadata.get("cascade_truth_hash")
CASCADE_DEFAULTS_TRUTH_PATH_VALUE = cascade_defaults_metadata.get("cascade_truth_path")

if CASCADE_DEFAULTS_TRUTH_HASH is None:
    raise ValueError("cascade_defaults_metadata is missing 'cascade_truth_hash'.")
if CASCADE_DEFAULTS_TRUTH_PATH_VALUE is None:
    raise ValueError("cascade_defaults_metadata is missing 'cascade_truth_path'.")

CASCADE_DEFAULTS_TRUTH_PATH = Path(CASCADE_DEFAULTS_TRUTH_PATH_VALUE)
if not CASCADE_DEFAULTS_TRUTH_PATH.exists():
    raise FileNotFoundError(f"Cascade defaults truth file not found: {CASCADE_DEFAULTS_TRUTH_PATH}")

cascade_defaults_truth = load_json(CASCADE_DEFAULTS_TRUTH_PATH)

if cascade_defaults_truth.get("truth_hash") != CASCADE_DEFAULTS_TRUTH_HASH:
    raise ValueError(
        "Cascade defaults metadata truth hash does not match the loaded cascade defaults truth file:\n"
        f"metadata={CASCADE_DEFAULTS_TRUTH_HASH}\n"
        f"truth_file={cascade_defaults_truth.get('truth_hash')}"
    )

cascade_defaults_results_truth_hash = extract_truth_hash(cascade_defaults_results)
if cascade_defaults_results_truth_hash is None:
    raise ValueError("Could not resolve meta__truth_hash from cascade_defaults_results CSV.")

if cascade_defaults_results_truth_hash != CASCADE_DEFAULTS_TRUTH_HASH:
    raise ValueError(
        "Cascade defaults results CSV truth hash does not match cascade_defaults_metadata['cascade_truth_hash']:\n"
        f"csv={cascade_defaults_results_truth_hash}\n"
        f"metadata={CASCADE_DEFAULTS_TRUTH_HASH}"
    )

# -------------------------------------------------------------------
# Resolve and validate Cascade Tuned stage truth
# -------------------------------------------------------------------
CASCADE_TUNED_TRUTH_HASH = cascade_tuned_metadata.get("cascade_truth_hash")
CASCADE_TUNED_TRUTH_PATH_VALUE = cascade_tuned_metadata.get("cascade_truth_path")

if CASCADE_TUNED_TRUTH_HASH is None:
    raise ValueError("cascade_tuned_metadata is missing 'cascade_truth_hash'.")
if CASCADE_TUNED_TRUTH_PATH_VALUE is None:
    raise ValueError("cascade_tuned_metadata is missing 'cascade_truth_path'.")

CASCADE_TUNED_TRUTH_PATH = Path(CASCADE_TUNED_TRUTH_PATH_VALUE)
if not CASCADE_TUNED_TRUTH_PATH.exists():
    raise FileNotFoundError(f"Cascade tuned truth file not found: {CASCADE_TUNED_TRUTH_PATH}")

cascade_tuned_truth = load_json(CASCADE_TUNED_TRUTH_PATH)

if cascade_tuned_truth.get("truth_hash") != CASCADE_TUNED_TRUTH_HASH:
    raise ValueError(
        "Cascade tuned metadata truth hash does not match the loaded cascade tuned truth file:\n"
        f"metadata={CASCADE_TUNED_TRUTH_HASH}\n"
        f"truth_file={cascade_tuned_truth.get('truth_hash')}"
    )

cascade_tuned_results_truth_hash = extract_truth_hash(cascade_tuned_results)
if cascade_tuned_results_truth_hash is None:
    raise ValueError("Could not resolve meta__truth_hash from cascade_tuned_results CSV.")

if cascade_tuned_results_truth_hash != CASCADE_TUNED_TRUTH_HASH:
    raise ValueError(
        "Cascade tuned results CSV truth hash does not match cascade_tuned_metadata['cascade_truth_hash']:\n"
        f"csv={cascade_tuned_results_truth_hash}\n"
        f"metadata={CASCADE_TUNED_TRUTH_HASH}"
    )

# -------------------------------------------------------------------
# Resolve and validate Cascade Stage 3 Improved stage truth
# -------------------------------------------------------------------
CASCADE_STAGE3_IMPROVED_TRUTH_HASH = cascade_stage3_improved_metadata.get("cascade_truth_hash")
CASCADE_STAGE3_IMPROVED_TRUTH_PATH_VALUE = cascade_stage3_improved_metadata.get("cascade_truth_path")

if CASCADE_STAGE3_IMPROVED_TRUTH_HASH is None:
    raise ValueError("cascade_stage3_improved_metadata is missing 'cascade_truth_hash'.")
if CASCADE_STAGE3_IMPROVED_TRUTH_PATH_VALUE is None:
    raise ValueError("cascade_stage3_improved_metadata is missing 'cascade_truth_path'.")

CASCADE_STAGE3_IMPROVED_TRUTH_PATH = Path(CASCADE_STAGE3_IMPROVED_TRUTH_PATH_VALUE)
if not CASCADE_STAGE3_IMPROVED_TRUTH_PATH.exists():
    raise FileNotFoundError(
        f"Cascade stage3 improved truth file not found: {CASCADE_STAGE3_IMPROVED_TRUTH_PATH}"
    )

cascade_stage3_improved_truth = load_json(CASCADE_STAGE3_IMPROVED_TRUTH_PATH)

if cascade_stage3_improved_truth.get("truth_hash") != CASCADE_STAGE3_IMPROVED_TRUTH_HASH:
    raise ValueError(
        "Cascade stage3 improved metadata truth hash does not match the loaded cascade stage3 improved truth file:\n"
        f"metadata={CASCADE_STAGE3_IMPROVED_TRUTH_HASH}\n"
        f"truth_file={cascade_stage3_improved_truth.get('truth_hash')}"
    )

cascade_stage3_improved_results_truth_hash = extract_truth_hash(cascade_stage3_improved_results)
if cascade_stage3_improved_results_truth_hash is None:
    raise ValueError("Could not resolve meta__truth_hash from cascade_stage3_improved_results CSV.")

if cascade_stage3_improved_results_truth_hash != CASCADE_STAGE3_IMPROVED_TRUTH_HASH:
    raise ValueError(
        "Cascade stage3 improved results CSV truth hash does not match cascade_stage3_improved_metadata['cascade_truth_hash']:\n"
        f"csv={cascade_stage3_improved_results_truth_hash}\n"
        f"metadata={CASCADE_STAGE3_IMPROVED_TRUTH_HASH}"
    )

# -------------------------------------------------------------------
# Shared parent truth validation
# -------------------------------------------------------------------
DEFAULTS_PARENT_GOLD_TRUTH_HASH = get_parent_truth_hash(cascade_defaults_truth)
TUNED_PARENT_GOLD_TRUTH_HASH = get_parent_truth_hash(cascade_tuned_truth)
STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH = get_parent_truth_hash(cascade_stage3_improved_truth)
BASELINE_PARENT_GOLD_TRUTH_HASH = get_parent_truth_hash(baseline_truth)

lineage_df = pd.DataFrame(
    [
        {
            "model_id": "baseline",
            "dataset_name": get_dataset_name_from_truth(baseline_truth),
            "stage_truth_hash": BASELINE_TRUTH_HASH,
            "parent_gold_truth_hash": BASELINE_PARENT_GOLD_TRUTH_HASH,
            "truth_path": str(BASELINE_TRUTH_PATH),
            "results_truth_hash": baseline_results_truth_hash,
            "result_rows": int(len(baseline_results)),
        },
        {
            "model_id": "cascade_default",
            "dataset_name": get_dataset_name_from_truth(cascade_defaults_truth),
            "stage_truth_hash": CASCADE_DEFAULTS_TRUTH_HASH,
            "parent_gold_truth_hash": DEFAULTS_PARENT_GOLD_TRUTH_HASH,
            "truth_path": str(CASCADE_DEFAULTS_TRUTH_PATH),
            "results_truth_hash": cascade_defaults_results_truth_hash,
            "result_rows": int(len(cascade_defaults_results)),
        },
        {
            "model_id": "cascade_tuned",
            "dataset_name": get_dataset_name_from_truth(cascade_tuned_truth),
            "stage_truth_hash": CASCADE_TUNED_TRUTH_HASH,
            "parent_gold_truth_hash": TUNED_PARENT_GOLD_TRUTH_HASH,
            "truth_path": str(CASCADE_TUNED_TRUTH_PATH),
            "results_truth_hash": cascade_tuned_results_truth_hash,
            "result_rows": int(len(cascade_tuned_results)),
        },
        {
            "model_id": "cascade_stage3_improved",
            "dataset_name": get_dataset_name_from_truth(cascade_stage3_improved_truth),
            "stage_truth_hash": CASCADE_STAGE3_IMPROVED_TRUTH_HASH,
            "parent_gold_truth_hash": STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH,
            "truth_path": str(CASCADE_STAGE3_IMPROVED_TRUTH_PATH),
            "results_truth_hash": cascade_stage3_improved_results_truth_hash,
            "result_rows": int(len(cascade_stage3_improved_results)),
        },
    ]
)

print("Comparison lineage summary:")
display(lineage_df)

DATASET_NAME = get_dataset_name_from_truth(baseline_truth)
CASCADE_DEFAULTS_DATASET_NAME = get_dataset_name_from_truth(cascade_defaults_truth)
CASCADE_TUNED_DATASET_NAME = get_dataset_name_from_truth(cascade_tuned_truth)
CASCADE_STAGE3_IMPROVED_DATASET_NAME = get_dataset_name_from_truth(cascade_stage3_improved_truth)

if DATASET_NAME != CASCADE_DEFAULTS_DATASET_NAME:
    raise ValueError(
        "Baseline and Cascade Defaults do not share the same dataset_name:\n"
        f"baseline_dataset={DATASET_NAME}\n"
        f"cascade_defaults_dataset={CASCADE_DEFAULTS_DATASET_NAME}"
    )

if DATASET_NAME != CASCADE_TUNED_DATASET_NAME:
    raise ValueError(
        "Baseline and Cascade Tuned do not share the same dataset_name:\n"
        f"baseline_dataset={DATASET_NAME}\n"
        f"cascade_tuned_dataset={CASCADE_TUNED_DATASET_NAME}"
    )

if DATASET_NAME != CASCADE_STAGE3_IMPROVED_DATASET_NAME:
    raise ValueError(
        "Baseline and Cascade Stage3 Improved do not share the same dataset_name:\n"
        f"baseline_dataset={DATASET_NAME}\n"
        f"cascade_stage3_improved_dataset={CASCADE_STAGE3_IMPROVED_DATASET_NAME}"
    )

COMPARISON_PARENT_GOLD_TRUTH_HASH = DEFAULTS_PARENT_GOLD_TRUTH_HASH

if COMPARISON_PARENT_GOLD_TRUTH_HASH is None:
    raise ValueError(
        "Cascade defaults truth is missing a usable parent_gold_truth_hash. "
        "Comparison lineage cannot be resolved."
    )

if TUNED_PARENT_GOLD_TRUTH_HASH != COMPARISON_PARENT_GOLD_TRUTH_HASH:
    raise ValueError(
        "Cascade tuned does not share the same parent Gold truth hash as cascade defaults.\n"
        f"cascade_defaults_parent={COMPARISON_PARENT_GOLD_TRUTH_HASH}\n"
        f"cascade_tuned_parent={TUNED_PARENT_GOLD_TRUTH_HASH}"
    )

if STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH != COMPARISON_PARENT_GOLD_TRUTH_HASH:
    raise ValueError(
        "Cascade stage3 improved does not share the same parent Gold truth hash as cascade defaults.\n"
        f"cascade_defaults_parent={COMPARISON_PARENT_GOLD_TRUTH_HASH}\n"
        f"cascade_stage3_improved_parent={STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH}"
    )

if BASELINE_PARENT_GOLD_TRUTH_HASH != COMPARISON_PARENT_GOLD_TRUTH_HASH:
    raise ValueError(
        "Baseline artifacts are from a different Gold lineage than the cascade artifacts used for comparison.\n"
        f"baseline_parent={BASELINE_PARENT_GOLD_TRUTH_HASH}\n"
        f"comparison_parent={COMPARISON_PARENT_GOLD_TRUTH_HASH}\n"
        f"baseline_truth_path={BASELINE_TRUTH_PATH}\n"
        f"cascade_defaults_truth_path={CASCADE_DEFAULTS_TRUTH_PATH}\n\n"
        "Re-run the baseline notebook from the same Gold parent lineage as the cascade runs, "
        "or repoint the baseline metadata/results to a matching baseline artifact set."
    )

GOLD_PARENT_TRUTH_HASH = COMPARISON_PARENT_GOLD_TRUTH_HASH

PIPELINE_MODE_FROM_BASELINE_TRUTH = get_pipeline_mode_from_truth(baseline_truth)
if PIPELINE_MODE_FROM_BASELINE_TRUTH is not None:
    PIPELINE_MODE = PIPELINE_MODE_FROM_BASELINE_TRUTH

GOLD_TRUTH_PATH = (
    TRUTHS_PATH
    / "gold"
    / f"{DATASET_NAME}__gold__truth__{GOLD_PARENT_TRUTH_HASH}.json"
)

if not GOLD_TRUTH_PATH.exists():
    raise FileNotFoundError(f"Gold truth file not found: {GOLD_TRUTH_PATH}")

gold_truth = load_json(GOLD_TRUTH_PATH)
gold_truth_runtime_facts = gold_truth.get("runtime_facts", {})
gold_truth_artifact_paths = gold_truth.get("artifact_paths", {})

logger.info("Resolved comparison dataset name from truth: %s", DATASET_NAME)
logger.info("Resolved comparison Gold parent truth hash: %s", GOLD_PARENT_TRUTH_HASH)

print("Comparison dataset name from truth:", DATASET_NAME)
print("Comparison Gold parent truth hash:", GOLD_PARENT_TRUTH_HASH)

ledger.add(
    kind="step",
    step="load_comparison_inputs",
    message=(
        "Loaded baseline/cascade outputs, validated their stage truth records, "
        "resolved the cascade comparison lineage, and loaded the shared Gold truth file."
    ),
    data={
        "baseline_results_path_csv": str(BASELINE_RESULTS_PATH_CSV),
        "baseline_results_path_pickle": str(BASELINE_RESULTS_PATH_PICKLE),
        "baseline_summary_path": str(BASELINE_SUMMARY_PATH),
        "baseline_thresholds_path": str(BASELINE_THRESHOLDS_PATH),
        "baseline_metadata_path": str(BASELINE_METADATA_PATH),
        "baseline_truth_hash": BASELINE_TRUTH_HASH,
        "baseline_truth_path": str(BASELINE_TRUTH_PATH),
        "baseline_results_truth_hash": baseline_results_truth_hash,
        "baseline_parent_gold_truth_hash": BASELINE_PARENT_GOLD_TRUTH_HASH,
        "cascade_defaults_results_path_csv": str(CASCADE_DEFAULTS_RESULTS_PATH_CSV),
        "cascade_defaults_results_path_pickle": str(CASCADE_DEFAULTS_RESULTS_PATH_PICKLE),
        "cascade_defaults_summary_path": str(CASCADE_DEFAULTS_SUMMARY_PATH),
        "cascade_defaults_thresholds_path": str(CASCADE_DEFAULTS_THRESHOLDS_PATH),
        "cascade_defaults_metadata_path": str(CASCADE_DEFAULTS_METADATA_PATH),
        "cascade_defaults_truth_hash": CASCADE_DEFAULTS_TRUTH_HASH,
        "cascade_defaults_truth_path": str(CASCADE_DEFAULTS_TRUTH_PATH),
        "cascade_defaults_results_truth_hash": cascade_defaults_results_truth_hash,
        "cascade_defaults_parent_gold_truth_hash": DEFAULTS_PARENT_GOLD_TRUTH_HASH,
        "cascade_tuned_results_path_csv": str(CASCADE_TUNED_RESULTS_PATH_CSV),
        "cascade_tuned_results_path_pickle": str(CASCADE_TUNED_RESULTS_PATH_PICKLE),
        "cascade_tuned_summary_path": str(CASCADE_TUNED_SUMMARY_PATH),
        "cascade_tuned_thresholds_path": str(CASCADE_TUNED_THRESHOLDS_PATH),
        "cascade_tuned_metadata_path": str(CASCADE_TUNED_METADATA_PATH),
        "cascade_tuned_truth_hash": CASCADE_TUNED_TRUTH_HASH,
        "cascade_tuned_truth_path": str(CASCADE_TUNED_TRUTH_PATH),
        "cascade_tuned_results_truth_hash": cascade_tuned_results_truth_hash,
        "cascade_tuned_parent_gold_truth_hash": TUNED_PARENT_GOLD_TRUTH_HASH,
        "cascade_stage3_improved_summary_path": str(CASCADE_STAGE3_IMPROVED_SUMMARY_PATH),
        "cascade_stage3_improved_thresholds_path": str(CASCADE_STAGE3_IMPROVED_THRESHOLDS_PATH),
        "cascade_stage3_improved_metadata_path": str(CASCADE_STAGE3_IMPROVED_METADATA_PATH),
        "cascade_stage3_improved_truth_hash": CASCADE_STAGE3_IMPROVED_TRUTH_HASH,
        "cascade_stage3_improved_truth_path": str(CASCADE_STAGE3_IMPROVED_TRUTH_PATH),
        "cascade_stage3_improved_results_truth_hash": cascade_stage3_improved_results_truth_hash,
        "cascade_stage3_improved_parent_gold_truth_hash": STAGE3_IMPROVED_PARENT_GOLD_TRUTH_HASH,
        "gold_truth_path": str(GOLD_TRUTH_PATH),
        "gold_truth_hash": GOLD_PARENT_TRUTH_HASH,
        "baseline_result_rows": int(len(baseline_results)),
        "cascade_defaults_result_rows": int(len(cascade_defaults_results)),
        "cascade_tuned_result_rows": int(len(cascade_tuned_results)),
        "cascade_stage3_improved_result_rows": int(len(cascade_stage3_improved_results)),
    },
    logger=logger,
)


baseline_results.head(3)

In [ ]:
if PLOT_CONTEXT == "baseline":
    if "baseline_results" in globals():
        plot_source_dataframe = baseline_results.copy()
    elif "gold_preprocessed_scaled_dataframe" in globals():
        plot_source_dataframe = gold_preprocessed_scaled_dataframe.copy()
    else:
        raise ValueError("Could not find baseline_results or gold_preprocessed_scaled_dataframe in memory.")
else:
    if "cascade_results" in globals():
        plot_source_dataframe = cascade_results.copy()
    elif "gold_preprocessed_scaled_dataframe" in globals():
        plot_source_dataframe = gold_preprocessed_scaled_dataframe.copy()
    else:
        raise ValueError("Could not find cascade_results or gold_preprocessed_scaled_dataframe in memory.")

plot_source_dataframe = ensure_stable_row_id(
    plot_source_dataframe,
    row_id_column=ROW_ID_COLUMN,
)

print(f"Plot source shape: {plot_source_dataframe.shape}")
display(plot_source_dataframe.head(3))

In [ ]:
def resolve_time_axis_column(
    dataframe: pd.DataFrame,
    candidates: list[str] | tuple[str, ...],
) -> str:
    for column_name in candidates:
        if column_name in dataframe.columns:
            return column_name
    raise ValueError(f"None of the requested time-axis columns were found: {candidates}")


def resolve_sensor_columns(
    dataframe: pd.DataFrame,
    sensor_prefixes: tuple[str, ...] = ("sensor_",),
) -> list[str]:
    sensor_columns = []
    for column_name in dataframe.columns:
        if any(column_name.startswith(prefix) for prefix in sensor_prefixes):
            if pd.api.types.is_numeric_dtype(dataframe[column_name]):
                sensor_columns.append(column_name)
    return sorted(sensor_columns)


TIME_AXIS_COLUMN = resolve_time_axis_column(
    plot_source_dataframe,
    TIME_AXIS_CANDIDATES,
)

SENSOR_COLUMNS = resolve_sensor_columns(
    plot_source_dataframe,
    sensor_prefixes=DEFAULT_SENSOR_PREFIXES,
)

print(f"Resolved time axis column: {TIME_AXIS_COLUMN}")
print(f"Sensor column count: {len(SENSOR_COLUMNS)}")
print(SENSOR_COLUMNS[:10])

In [ ]:
plot_dataframe = plot_source_dataframe.copy()

if np.issubdtype(plot_dataframe[TIME_AXIS_COLUMN].dtype, np.datetime64):
    plot_dataframe = plot_dataframe.sort_values(by=[TIME_AXIS_COLUMN, ROW_ID_COLUMN]).reset_index(drop=True)
else:
    plot_dataframe = plot_dataframe.sort_values(by=[TIME_AXIS_COLUMN, ROW_ID_COLUMN]).reset_index(drop=True)

plot_dataframe["plot_order_index"] = np.arange(len(plot_dataframe), dtype=np.int64)

display(
    plot_dataframe[
        [col for col in [ROW_ID_COLUMN, TIME_AXIS_COLUMN, "plot_order_index", "time_index", "event_step", "event_time"] if col in plot_dataframe.columns]
    ].head(10)
)

In [ ]:
baseline_detected_rows_dataframe = pd.DataFrame()
stage1_detected_rows_dataframe = pd.DataFrame()
stage2_detected_rows_dataframe = pd.DataFrame()
final_detected_rows_dataframe = pd.DataFrame()

if PLOT_CONTEXT == "baseline":
    if "baseline_flag" not in plot_dataframe.columns:
        raise ValueError("baseline_flag is missing from the plotting dataframe.")

    baseline_detected_rows_dataframe = get_stage_detected_rows_dataframe(
        plot_dataframe,
        stage_name="baseline",
        row_id_column=ROW_ID_COLUMN,
        include_columns=["baseline_flag", "anomaly_flag"] if "anomaly_flag" in plot_dataframe.columns else ["baseline_flag"],
        sort_by=TIME_AXIS_COLUMN if TIME_AXIS_COLUMN in plot_dataframe.columns else "time_index",
        ascending=True,
    )

    print(f"Baseline detected rows: {len(baseline_detected_rows_dataframe):,}")
    display(baseline_detected_rows_dataframe.head(10))

else:
    if "stage1_flag" in plot_dataframe.columns:
        stage1_detected_rows_dataframe = get_stage_detected_rows_dataframe(
            plot_dataframe,
            stage_name="stage1",
            row_id_column=ROW_ID_COLUMN,
            include_columns=[
                col for col in ["stage1_flag", "stage2_flag", "cascade_final_flag", "anomaly_flag"]
                if col in plot_dataframe.columns
            ],
            sort_by=TIME_AXIS_COLUMN if TIME_AXIS_COLUMN in plot_dataframe.columns else "time_index",
            ascending=True,
        )

    if "stage2_flag" in plot_dataframe.columns:
        stage2_detected_rows_dataframe = get_stage_detected_rows_dataframe(
            plot_dataframe,
            stage_name="stage2",
            row_id_column=ROW_ID_COLUMN,
            include_columns=[
                col for col in [
                    "stage2_raw_flag",
                    "stage2_flag",
                    "cascade_final_flag",
                    "stage2_model_score",
                    "stage2_model_decision",
                    "stage2_model_pred",
                    "stage2_model_flag",
                    "anomaly_flag",
                ] if col in plot_dataframe.columns
            ],
            sort_by=TIME_AXIS_COLUMN if TIME_AXIS_COLUMN in plot_dataframe.columns else "time_index",
            ascending=True,
        )

    if "cascade_final_flag" in plot_dataframe.columns:
        final_detected_rows_dataframe = get_stage_detected_rows_dataframe(
            plot_dataframe,
            stage_name="cascade_final",
            row_id_column=ROW_ID_COLUMN,
            include_columns=[
                col for col in [
                    "stage1_flag",
                    "stage2_raw_flag",
                    "stage2_flag",
                    "cascade_final_flag",
                    "stage3_profile_breach_count",
                    "stage3_secondary_breach_count",
                    "stage3_persistence_flag",
                    "stage3_drift_flag",
                    "stage3_rule_evidence_count",
                    "anomaly_flag",
                ] if col in plot_dataframe.columns
            ],
            sort_by=TIME_AXIS_COLUMN if TIME_AXIS_COLUMN in plot_dataframe.columns else "time_index",
            ascending=True,
        )

    print(f"Stage 1 detected rows: {len(stage1_detected_rows_dataframe):,}")
    print(f"Stage 2 detected rows: {len(stage2_detected_rows_dataframe):,}")
    print(f"Final cascade detected rows: {len(final_detected_rows_dataframe):,}")

In [ ]:
def attach_plot_order_index(
    base_dataframe: pd.DataFrame,
    detected_dataframe: pd.DataFrame,
    *,
    row_id_column: str = ROW_ID_COLUMN,
) -> pd.DataFrame:
    if detected_dataframe is None or detected_dataframe.empty:
        return pd.DataFrame()

    join_columns = [row_id_column, "plot_order_index"]
    if TIME_AXIS_COLUMN in base_dataframe.columns and TIME_AXIS_COLUMN not in join_columns:
        join_columns.append(TIME_AXIS_COLUMN)

    enriched = detected_dataframe.merge(
        base_dataframe[join_columns].drop_duplicates(subset=[row_id_column]),
        on=row_id_column,
        how="left",
    )

    if "plot_order_index" in enriched.columns:
        enriched = enriched.sort_values(by="plot_order_index").reset_index(drop=True)

    return enriched


baseline_detected_rows_plot = attach_plot_order_index(plot_dataframe, baseline_detected_rows_dataframe)
stage1_detected_rows_plot = attach_plot_order_index(plot_dataframe, stage1_detected_rows_dataframe)
stage2_detected_rows_plot = attach_plot_order_index(plot_dataframe, stage2_detected_rows_dataframe)
final_detected_rows_plot = attach_plot_order_index(plot_dataframe, final_detected_rows_dataframe)

display(baseline_detected_rows_plot.head(3) if not baseline_detected_rows_plot.empty else pd.DataFrame())
display(stage2_detected_rows_plot.head(3) if not stage2_detected_rows_plot.empty else pd.DataFrame())
display(final_detected_rows_plot.head(3) if not final_detected_rows_plot.empty else pd.DataFrame())

In [ ]:
def plot_sensor_time_series_with_anomalies(
    dataframe: pd.DataFrame,
    *,
    sensor_column: str,
    context: str = "cascade",
    time_axis_column: str = TIME_AXIS_COLUMN,
    row_id_column: str = ROW_ID_COLUMN,
    baseline_detected_rows: pd.DataFrame | None = None,
    stage1_detected_rows: pd.DataFrame | None = None,
    stage2_detected_rows: pd.DataFrame | None = None,
    final_detected_rows: pd.DataFrame | None = None,
    figsize: tuple[int, int] = (16, 6),
    show_score_axis: bool = True,
    save_plot: bool = False,
):
    if sensor_column not in dataframe.columns:
        raise ValueError(f"Missing sensor column: {sensor_column}")

    fig, ax1 = plt.subplots(figsize=figsize)

    x_values = dataframe["plot_order_index"]
    y_values = pd.to_numeric(dataframe[sensor_column], errors="coerce")

    ax1.plot(x_values, y_values, linewidth=1.2)
    ax1.set_title(f"{sensor_column} — {context} time series with tracked rows")
    ax1.set_xlabel(f"plot_order_index (ordered by {time_axis_column})")
    ax1.set_ylabel(sensor_column)

    # Optional true label overlay
    if "anomaly_flag" in dataframe.columns:
        true_anomaly_rows = dataframe.loc[dataframe["anomaly_flag"].fillna(0).astype(int) == 1].copy()
        if not true_anomaly_rows.empty:
            ax1.scatter(
                true_anomaly_rows["plot_order_index"],
                pd.to_numeric(true_anomaly_rows[sensor_column], errors="coerce"),
                s=18,
                alpha=0.6,
                label="true anomaly_flag",
            )

    if context == "baseline":
        if baseline_detected_rows is not None and not baseline_detected_rows.empty:
            baseline_hits = baseline_detected_rows.merge(
                dataframe[[row_id_column, "plot_order_index", sensor_column]],
                on=[row_id_column, "plot_order_index"],
                how="left",
            )
            ax1.scatter(
                baseline_hits["plot_order_index"],
                pd.to_numeric(baseline_hits[sensor_column], errors="coerce"),
                s=ANOMALY_MARKER_SIZE,
                alpha=0.9,
                label="baseline detected",
            )

        if show_score_axis and "baseline_score" in dataframe.columns:
            ax2 = ax1.twinx()
            ax2.plot(
                x_values,
                pd.to_numeric(dataframe["baseline_score"], errors="coerce"),
                alpha=0.8,
                linestyle="--",
            )
            ax2.set_ylabel("baseline_score")

    else:
        if stage1_detected_rows is not None and not stage1_detected_rows.empty:
            stage1_hits = stage1_detected_rows.merge(
                dataframe[[row_id_column, "plot_order_index", sensor_column]],
                on=[row_id_column, "plot_order_index"],
                how="left",
            )
            ax1.scatter(
                stage1_hits["plot_order_index"],
                pd.to_numeric(stage1_hits[sensor_column], errors="coerce"),
                s=STAGE1_MARKER_SIZE,
                alpha=0.6,
                label="stage1 detected",
            )

        if stage2_detected_rows is not None and not stage2_detected_rows.empty:
            stage2_hits = stage2_detected_rows.merge(
                dataframe[[row_id_column, "plot_order_index", sensor_column]],
                on=[row_id_column, "plot_order_index"],
                how="left",
            )
            ax1.scatter(
                stage2_hits["plot_order_index"],
                pd.to_numeric(stage2_hits[sensor_column], errors="coerce"),
                s=STAGE2_MARKER_SIZE,
                alpha=0.8,
                label="stage2 detected",
            )

        if final_detected_rows is not None and not final_detected_rows.empty:
            final_hits = final_detected_rows.merge(
                dataframe[[row_id_column, "plot_order_index", sensor_column]],
                on=[row_id_column, "plot_order_index"],
                how="left",
            )
            ax1.scatter(
                final_hits["plot_order_index"],
                pd.to_numeric(final_hits[sensor_column], errors="coerce"),
                s=FINAL_MARKER_SIZE,
                alpha=0.95,
                label="final cascade alert",
            )

        if show_score_axis and "stage2_score" in dataframe.columns:
            ax2 = ax1.twinx()
            ax2.plot(
                x_values,
                pd.to_numeric(dataframe["stage2_score"], errors="coerce"),
                alpha=0.8,
                linestyle="--",
            )
            ax2.set_ylabel("stage2_score")

    handles, labels = ax1.get_legend_handles_labels()
    if len(handles) > 0:
        ax1.legend(loc="best")

    plt.tight_layout()

    if save_plot:
        output_path = PLOT_OUTPUT_DIR / f"{context}_{sensor_column}_time_series.png"
        plt.savefig(output_path, bbox_inches="tight")
        print(f"Saved plot to: {output_path}")

    plt.show()

In [ ]:
SENSOR_TO_PLOT = SENSOR_COLUMNS[0] if SENSOR_COLUMNS else None

if SENSOR_TO_PLOT is None:
    raise ValueError("No sensor columns were found for plotting.")

plot_sensor_time_series_with_anomalies(
    plot_dataframe,
    sensor_column=SENSOR_TO_PLOT,
    context=PLOT_CONTEXT,
    baseline_detected_rows=baseline_detected_rows_plot,
    stage1_detected_rows=stage1_detected_rows_plot,
    stage2_detected_rows=stage2_detected_rows_plot,
    final_detected_rows=final_detected_rows_plot,
    show_score_axis=True,
    save_plot=SAVE_PLOTS,
)

In [ ]:
SENSORS_TO_PLOT = SENSOR_COLUMNS[:6]

for sensor_name in SENSORS_TO_PLOT:
    plot_sensor_time_series_with_anomalies(
        plot_dataframe,
        sensor_column=sensor_name,
        context=PLOT_CONTEXT,
        baseline_detected_rows=baseline_detected_rows_plot,
        stage1_detected_rows=stage1_detected_rows_plot,
        stage2_detected_rows=stage2_detected_rows_plot,
        final_detected_rows=final_detected_rows_plot,
        show_score_axis=True,
        save_plot=SAVE_PLOTS,
    )

In [ ]:
def chunk_list(values: list[str], chunk_size: int) -> list[list[str]]:
    return [values[i:i + chunk_size] for i in range(0, len(values), chunk_size)]


sensor_batches = chunk_list(SENSOR_COLUMNS, MAX_SENSORS_PER_BATCH)

print(f"Total batches: {len(sensor_batches)}")
print(sensor_batches[:2])

In [ ]:
BATCH_INDEX = 0
CURRENT_BATCH = sensor_batches[BATCH_INDEX]

for sensor_name in CURRENT_BATCH:
    plot_sensor_time_series_with_anomalies(
        plot_dataframe,
        sensor_column=sensor_name,
        context=PLOT_CONTEXT,
        baseline_detected_rows=baseline_detected_rows_plot,
        stage1_detected_rows=stage1_detected_rows_plot,
        stage2_detected_rows=stage2_detected_rows_plot,
        final_detected_rows=final_detected_rows_plot,
        show_score_axis=True,
        save_plot=SAVE_PLOTS,
    )

In [ ]:
def build_window_around_detected_rows(
    dataframe: pd.DataFrame,
    detected_rows_dataframe: pd.DataFrame,
    *,
    row_id_column: str = ROW_ID_COLUMN,
    window_radius: int = 50,
) -> pd.DataFrame:
    if detected_rows_dataframe is None or detected_rows_dataframe.empty:
        return pd.DataFrame()

    detected_with_order = detected_rows_dataframe.merge(
        dataframe[[row_id_column, "plot_order_index"]],
        on=row_id_column,
        how="left",
    )

    indices_to_keep = set()
    for idx in detected_with_order["plot_order_index"].dropna().astype(int).tolist():
        start_idx = max(0, idx - window_radius)
        end_idx = min(len(dataframe) - 1, idx + window_radius)
        indices_to_keep.update(range(start_idx, end_idx + 1))

    zoom_frame = dataframe.loc[dataframe["plot_order_index"].isin(sorted(indices_to_keep))].copy()
    return zoom_frame.sort_values("plot_order_index").reset_index(drop=True)

In [ ]:
if PLOT_CONTEXT == "cascade" and not final_detected_rows_plot.empty:
    zoom_dataframe = build_window_around_detected_rows(
        plot_dataframe,
        final_detected_rows_plot,
        window_radius=40,
    )

    print(f"Zoom dataframe shape: {zoom_dataframe.shape}")

    plot_sensor_time_series_with_anomalies(
        zoom_dataframe,
        sensor_column=SENSOR_COLUMNS[0],
        context="cascade",
        baseline_detected_rows=pd.DataFrame(),
        stage1_detected_rows=attach_plot_order_index(zoom_dataframe, stage1_detected_rows_plot),
        stage2_detected_rows=attach_plot_order_index(zoom_dataframe, stage2_detected_rows_plot),
        final_detected_rows=attach_plot_order_index(zoom_dataframe, final_detected_rows_plot),
        show_score_axis=True,
        save_plot=False,
    )

In [ ]:
EXPORT_DIR = Path("artifacts/gold/exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if not baseline_detected_rows_plot.empty:
    baseline_detected_rows_plot.to_csv(EXPORT_DIR / "baseline_detected_rows_for_plotting.csv", index=False)

if not stage1_detected_rows_plot.empty:
    stage1_detected_rows_plot.to_csv(EXPORT_DIR / "cascade_stage1_detected_rows_for_plotting.csv", index=False)

if not stage2_detected_rows_plot.empty:
    stage2_detected_rows_plot.to_csv(EXPORT_DIR / "cascade_stage2_detected_rows_for_plotting.csv", index=False)

if not final_detected_rows_plot.empty:
    final_detected_rows_plot.to_csv(EXPORT_DIR / "cascade_final_detected_rows_for_plotting.csv", index=False)

print("Detected-row plotting exports written.")